## Импорты и параметры


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install -U transformers
!pip install -q trl bert-score openai
!pip install trl==0.11.3
!pip install -q accelerate>=1.8.0
!pip install -q bitsandbytes>=0.46.1

import time
import random
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datasets import Dataset
import os
import json

import torch
torch.cuda.empty_cache()

from trl import PPOConfig, PPOTrainer, AutoModelForCausalLMWithValueHead
from transformers import AutoTokenizer, BitsAndBytesConfig, Qwen2ForCausalLM
from bert_score import BERTScorer
import openai
import gc
from tqdm import tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.0/11.0 MB 93.7 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.9.0
    Uninstalling transformers-5.9.0:
      Successfully uninstalled transformers-5.9.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 761.1/761.1 kB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 50.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 21.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 316.6/316.6 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 20.6 MB/s eta 0:00:00
  Attempting uninstall: trl
    Found existing installation: trl 1.5.1
    Uninstalling trl-1.5.1:
      Successfully uninstalled trl-1.5.1


In [ ]:
# дообучение
USE_LLM_REWARD = True
MAX_INPUT_TOKENS = 3600
MAX_OUTPUT_TOKENS = 1024
MODEL_PATH = "/content/drive/MyDrive/ПЗАД проект/new_ppo/ppo_qwen2.5_1.5B_bs1_4bit"

# параметры LLM-score
OPENROUTER_API_KEY = ""
JUDGE_MODEL = "openai/gpt-oss-120b:free"
LLM_MAX_RETRIES = 5
LLM_BASE_DELAY = 1.0
last_llm_score = 0.0
LLM_CALL_EVERY_N_STEPS = 1

# BERTScore
bertscorer = BERTScorer(lang="ru", rescale_with_baseline=False, device='cpu')

# клиент OpenRouter
if USE_LLM_REWARD:
    openrouter_client = openai.OpenAI(
        base_url="https://openrouter.ai/api/v1",
        api_key=OPENROUTER_API_KEY,
        default_headers={"Content-Type": "application/json; charset=utf-8"}
    )

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/714M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


### Сохраняем рандом сид

In [ ]:
def set_random_seed(seed: int = 27):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.backends.cudnn.deterministic = True

set_random_seed()

## Загрузка модели и токенизатора

In [ ]:
print("Загрузка модели...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLMWithValueHead.from_pretrained(
    MODEL_PATH,
    device_map="auto",
    torch_dtype=torch.float16
)
model.eval()
print("Модель загружена")

Загрузка модели...


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Модель загружена


## Добавление промпта в тестовый датасет

In [ ]:
def build_prompt_ppo(example, tokenizer):
    """Формирует промт (query) как в исходном build_prompt, но без ответа"""
    instruction = f"""You are an expert in interview analysis. Your task is to highlight the thematic codes in the interview text and corresponding quotes.

Act step by step:

1. Carefully review the output format shown in the example below. Remember that each code block must contain "Общий код" (General code), then Quote, then "Конкретный код" (Specific code). The quote must be verbatim and enclosed in quotation marks.
2. Read the interview transcript and the topic. Identify all fragments (quotes) that relate to the interview topic.
3. Group the quotes by general themes — these will be the "Общий код" (General codes). For each general theme, come up with a short name.
4. Within each general code, identify specific meaning aspects — these will be the "конкретный код" (Specific codes). The names of specific codes should reflect the essence of the quote.
5. Generate the answer strictly following the format from the example. Do not add any explanations, do not write words like 'Step 1', 'Step 2' — only the final blocks of codes and quotes.

Format of an output (consists of several general codes, each followed by quotes and specific codes):
**Общий код 1: <generate general code 1>**
"<quote text>" - **<generate specific code 1> (Конкретный код)**
"<quote text>" - **<generate specific code 2> (Конкретный код)**
**Общий код 2: <generate general code 2>**
"<quote text>" - **<generate specific code> (Конкретный код)**
**Общий код <general code number>: <generate general code>**
"<quote text>" - **<generate specific code> (Конкретный код)**
and so on. You choose the number of general and specific codes.

Example of a topic:
ПОКОЛЕНЧЕСКАЯ ИДЕНТИЧНОСТЬ, ЖИЗНЕННЫЕ ВЫБОРЫ И УСТОЙЧИВОСТЬ РОССИЙСКОЙ МОЛОДЕЖИ

Example of a general code related to the topic (pay attention to the structure):
**Общий код 1: Поколенческие характеристики, ценности и жизненные ориентиры**
"Мне кажется, большинство особо не стремятся там вот срочно, прямо сейчас там жениться, замуж, там детей и так далее. Вот. То есть как-то больше сосредоточены даже не на карьере, а на себе, на том, чтобы сложить все для себя вот так, как хочется, да. То есть не просто чтобы там выйти замуж, а чтобы выйти замуж вот по любви, чтобы все было идеально. Вот на какой-то такой идеальности что ли." - **Фокус на самореализации и качественных отношениях (конкретный код)**
"У нашего поколения все-таки все по-другому. Нам не дадут квартиру просто так. Нам не обязательно так просто получить место там где-то на работе и так далее, да. Но при этом у нас гораздо больше возможностей в плане, как сказать, чему-то научиться новому, куда-то поехать, что-то посмотреть, составить свое мнение, там высказать свое мнение даже." - **Осознание свободы выбора и новых возможностей (конкретный код)**
"У детей нынешних у них как будто меньше табу что ли. [...] они спокойно со мной могла поговорить на какие-то откровенные темы, которые мне в ее возрасте, я тоже задумывалась об этом, у меня тоже было какое-то мнение, но я боялась об этом говорить со взрослыми, потому что это было табуировано. [...] Поэтому у них мне кажется растет какое-то более свободное поколение что ли." - **Сравнение с младшим поколением: свобода от табу (конкретный код)**
"Мне кажется, что поколение у нас достаточно трудолюбивое при этом как бы. То есть если человек чего-то хочет добиться в карьерной сфере, ну, человек действительно может приложить там все усилия и добиться этого. Вот. Потому что возможностей супер много сейчас." - **Трудолюбие и вера в возможности (конкретный код)**

Now you should do the markup for the interview according to the plan. Important: The answer should contain only codes and quotes, without unnecessary words and repetitions.
The quotes must be strictly from the text. Give the answer in Russian.
"""

    user_content = f"""Тема интервью:
{example['topic']}

Текст интервью:
{example['transcript']}

Пожалуйста, выполни разметку интервью, каждый общий код в указанном формате:
**Общий код <generate general code>: <general code name>**
"<quote text>" - **<generate specific code> (Конкретный код)**"""

    messages = [
        {"role": "system", "content": instruction},
        {"role": "user", "content": user_content},
    ]
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    return prompt

## Загрузка данных и добавление промпта

In [ ]:
test_df = pd.read_csv('test_data.csv')
test_dataset = Dataset.from_pandas(test_df)

In [ ]:
def prepare_dataset(hf_dataset, tokenizer):
    queries, targets = [], []
    topics, transcripts = [], []
    for ex in hf_dataset:
        queries.append(build_prompt_ppo(ex, tokenizer))
        targets.append(ex['coding'])
        topics.append(ex['topic'])
        transcripts.append(ex['transcript'])
    return Dataset.from_dict(
        {"query": queries, "coding": targets, "topic": topics, "transcript": transcripts}
    )

In [ ]:
test_dataset = prepare_dataset(test_dataset, tokenizer)

## Получение Reward

In [ ]:
llm_cache = {}

def get_llm_score(generated_text, source_text, topic):
    cache_key = (generated_text, source_text, topic)
    if cache_key in llm_cache:
        return llm_cache[cache_key]

    if not USE_LLM_REWARD:
        return None

    model = JUDGE_MODEL

    prompt = f"""Ты - социолог-эксперт в открытом кодировании интервью.
Тема интервью: {topic}
Текст интервью: {source_text}

Сгенерированные коды и цитаты (в формате <код>цитата</код>):
{generated_text}

Оцени результат по трём критериям (каждый от 0 до 1):
1. Релевантность - насколько код соответствует содержанию фрагмента интервью с учётом темы.
2. Когерентность - насколько формулировка кода грамматически корректна и стилистически приемлема.
3. Теоретический инсайт - степень соответствия кода концептуальным ожиданиям.

Ответ дай строго в формате: число, число, число (например: 0.85, 0.90, 0.75). Не пиши никаких пояснений."""

    max_retries = 5
    base_delay = 1.0
    max_delay = 30.0

    for attempt in range(max_retries):
        try:
            response = openrouter_client.chat.completions.create(
                model=model,
                messages=[{"role": "user", "content": prompt}],
                temperature=0.0,
                max_tokens=50
            )

            if response and hasattr(response, 'choices') and response.choices:
                content = response.choices[0].message.content
                if content:

                    numbers = re.findall(r"(\d+\.?\d*)", content)
                    if len(numbers) >= 3:
                        try:
                            scores = [float(n) for n in numbers[:3]]
                            mean_score = np.mean(scores)
                            mean_score = max(0.0, min(1.0, mean_score))
                            llm_cache[cache_key] = mean_score
                            return mean_score
                        except ValueError:
                            print(f"LLM warning: non-numeric numbers: {numbers}", flush=True)
                    else:
                        print(f"LLM warning: expected 3 numbers, got {len(numbers)}. Content: {content[:200]}", flush=True)
                else:
                    print("LLM warning: empty content", flush=True)
            else:
                print("LLM warning: invalid response structure", flush=True)

            return None

        except Exception as e:
            is_retryable = False
            if hasattr(e, 'status_code'):
                if e.status_code == 429 or (500 <= e.status_code < 600):
                    is_retryable = True
            elif '429' in str(e) or 'rate limit' in str(e).lower():
                is_retryable = True

            if not is_retryable or attempt == max_retries - 1:
                print(f"LLM error (final): {type(e).__name__}: {e}", flush=True)
                return None
            delay = min(base_delay * (2 ** attempt), max_delay)
            jitter = random.uniform(0, 0.1 * delay)
            wait_time = delay + jitter
            print(f"LLM error (retryable): {e}. Retrying in {wait_time:.2f}s... (attempt {attempt+1}/{max_retries})", flush=True)
            time.sleep(wait_time)

    return None

def compute_reward(generated, target, source_text, topic, use_llm=True):
    global last_llm_score
    _, _, bert_f1 = bertscorer.score([generated], [target])
    bert_reward = bert_f1.item()
    if use_llm:
        new_llm = get_llm_score(generated, source_text, topic)
        if new_llm is not None:
            last_llm_score = new_llm
        llm_reward = last_llm_score
    else:
        llm_reward = 0.0
    return bert_reward + llm_reward, bert_reward, llm_reward

## Инференс модели

### Функция генерации

In [ ]:
def generate_code(prompt, max_new_tokens=MAX_OUTPUT_TOKENS, num_beams=3, return_tensors=True):
    device = model.pretrained_model.device
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=MAX_INPUT_TOKENS).to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            num_beams=num_beams,
            early_stopping=True,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

    generated = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    response_ids = outputs[0, inputs['input_ids'].shape[1]:]
    if return_tensors:
        return generated, inputs, response_ids
    return generated

### Очистка CUDA

In [ ]:
def clear_cuda_cache():
    """Очистка кэша CUDA памяти"""
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
        gc.collect()
        allocated = torch.cuda.memory_allocated() / 1024**3
        reserved = torch.cuda.memory_reserved() / 1024**3
        return allocated, reserved
    return 0, 0

clear_cuda_cache()

print("Allocated:", round(torch.cuda.memory_allocated()/2**30, 2), "GB")
print("Reserved: ", round(torch.cuda.memory_reserved()/2**30, 2), "GB")
print("Free:     ", round((torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_reserved())/2**30, 2), "GB")

Allocated: 1.06 GB
Reserved:  1.08 GB
Free:      13.49 GB


### Запуск оценки на тестовой выборке

In [ ]:
device = next(model.parameters()).device
predictions = []
all_rewards_total = []
all_rewards_bert = []
all_rewards_llm = []

clear_cuda_cache()

for idx in tqdm(range(len(test_dataset)), desc="Evaluating"):
    batch_item = test_dataset[idx]
    query = batch_item["query"]
    target = batch_item["coding"]
    src = batch_item["transcript"]
    topic = batch_item["topic"]

    pred = generate_code(query, return_tensors=False)

    total_reward, bert_reward, llm_reward = compute_reward(pred, target, src, topic, use_llm=USE_LLM_REWARD)

    print(f'BERTScore: {bert_reward}; LLM score: {llm_reward}')

    predictions.append({
        "query": query,
        "prediction": pred,
        "target": target,
        "source_text": src,
        "topic": topic,
        "reward_total": total_reward,
        "reward_bert": bert_reward,
        "reward_llm": llm_reward
    })

pred_df = pd.DataFrame(predictions)

print("\n=== Test Metrics ===")

print(f'\nGeneration strategy: Beam Search')
print(f"Average Total Reward: {np.mean(pred_df['reward_total']):.4f}")
print(f"Average BERTScore: {np.mean(pred_df['reward_bert']):.4f}")
print(f"Average LLM Score: {np.mean(pred_df['reward_llm']):.4f}")

Evaluating:   4%|▍         | 1/23 [04:02<1:29:02, 242.82s/it]

BERTScore: 0.7268234491348267; LLM score: 0.43333333333333335


Evaluating:   9%|▊         | 2/23 [08:04<1:24:48, 242.32s/it]

BERTScore: 0.7148221135139465; LLM score: 0.8099999999999999


Evaluating:  13%|█▎        | 3/23 [12:07<1:20:48, 242.43s/it]

...

BERTScore: 0.7371167540550232; LLM score: 0.4366666666666666


Evaluating:  96%|█████████▌| 22/23 [1:26:45<04:00, 240.77s/it]

BERTScore: 0.7016471028327942; LLM score: 0.4333333333333333


Evaluating: 100%|██████████| 23/23 [1:30:48<00:00, 236.90s/it]

BERTScore: 0.6962192058563232; LLM score: 0.6

=== Test Metrics ===

Generation strategy: Beam Search
Average Total Reward: 1.2577
Average BERTScore: 0.7247
Average LLM Score: 0.5330


In [ ]:
pred_df.to_csv(f'{MODEL_PATH}/evaluation.csv', index=False)
print(f"Результаты сохранены в evaluation.csv")
pred_df.head()

Результаты сохранены в evaluation.csv


,query,prediction,target,source_text,topic,reward_total,reward_bert,reward_llm
0,<|im_start|>system\nYou are an expert in inter...,"**Общий код 1: Поколенческие характеристики, ц...",**Общий код 1: Трудовой путь в спорте: мотивац...,"Информант: …Заработная плата маленькая, а мы в...",Тема: ПОКОЛЕНИЕ Z В ПОИСКАХ БАЛАНСА: УСЛОВИЯ Т...,1.160157,0.726823,0.433333
1,<|im_start|>system\nYou are an expert in inter...,"**Общий код 1: Поколенческие характеристики, ц...","**Общий код 1: Ранний трудовой опыт, мотивация...",Интервьюер: Меня зовут [имя]. И сегодня мы пог...,Тема: ПОКОЛЕНИЕ Z В ПОИСКАХ БАЛАНСА: УСЛОВИЯ Т...,1.524822,0.714822,0.810000
2,<|im_start|>system\nYou are an expert in inter...,**Общий код 1: Самостоятельность и инициативно...,**Общий код 1: Мотивация и путь в проактивную ...,"Инт: Сегодня у нас первое ноль седьмое, и, со...",Тема: СОЗИДАТЕЛЬНЫЙ ПОТЕНЦИАЛ САМОСТОЯТЕЛЬНОГО...,1.403903,0.727236,0.676667
3,<|im_start|>system\nYou are an expert in inter...,"**Общий код 1: Поколенческие характеристики, ц...","**Общий код 1: Поколенческие характеристики, ц...","Интервьюер: Тогда, наверное, начнем. Сперва мо...","Тема: ПОКОЛЕНЧЕСКАЯ ИДЕНТИЧНОСТЬ, ЖИЗНЕННЫЕ ВЫ...",1.237431,0.737431,0.500000
4,<|im_start|>system\nYou are an expert in inter...,"**Общий код 1: Поколенческие характеристики, ц...",**Общий код 1: Представления о работе и критер...,"Интервьюер: Давай тогда первый вопрос, можешь ...",Тема: ПОКОЛЕНИЕ Z В ПОИСКАХ БАЛАНСА: УСЛОВИЯ Т...,1.270844,0.720844,0.550000
